In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from robotblockset.tools import get_rbs_path, print_xml, print_xml_for_console, find_attr_values_in_xml

import mujoco 
from robotblockset.mujoco.tools_mjcf import print_body_tree, actuators_for_joint, replace_in_mjcf_file

In [3]:
PLATFORM = "unitree_b2"
PLATFORM_NAME = "b2"
ROBOT = "z1"
ROBOT_NAME = "z1"
CAMERA = None

In [4]:
MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/"
print(MODEL_PATH)

d:\Leon\Python\RBS\robotblockset/mujoco/mjcf_models/


In [5]:
scene=mujoco.MjSpec.from_file(MODEL_PATH + "scene.xml")
scene.copy_during_attach = True
# print_xml(scene.to_xml())

In [6]:
type(scene)

mujoco._specs.MjSpec

In [7]:
platform_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{PLATFORM}.xml")
platform_spec.copy_during_attach = True
print_body_tree(platform_spec.worldbody, platform_spec)
# print_xml(platform_spec.to_xml())

Body Tree for "world"
-base (Joints: floating_base-FREE)
--FL_hip (Joints: FL_hip_joint-HINGE[Actuator: pos_FL_hip_joint])
---FL_thigh (Joints: FL_thigh_joint-HINGE[Actuator: pos_FL_thigh_joint])
----FL_calf (Joints: FL_calf_joint-HINGE[Actuator: pos_FL_calf_joint])
-----FL_foot
----FL_thigh_protect
--FR_hip (Joints: FR_hip_joint-HINGE[Actuator: pos_FR_hip_joint])
---FR_thigh (Joints: FR_thigh_joint-HINGE[Actuator: pos_FR_thigh_joint])
----FR_calf (Joints: FR_calf_joint-HINGE[Actuator: pos_FR_calf_joint])
-----FR_foot
----FR_thigh_protect
--RL_hip (Joints: RL_hip_joint-HINGE[Actuator: pos_RL_hip_joint])
---RL_thigh (Joints: RL_thigh_joint-HINGE[Actuator: pos_RL_thigh_joint])
----RL_calf (Joints: RL_calf_joint-HINGE[Actuator: pos_RL_calf_joint])
-----RL_foot
----RL_thigh_protect
--RR_hip (Joints: RR_hip_joint-HINGE[Actuator: pos_RR_hip_joint])
---RR_thigh (Joints: RR_thigh_joint-HINGE[Actuator: pos_RR_thigh_joint])
----RR_calf (Joints: RR_calf_joint-HINGE[Actuator: pos_RR_calf_joint])
-

Prepare data for model keyframes

In [8]:
if len(platform_spec.keys) > 0:
    platform_ctrl = np.array(platform_spec.keys[0].ctrl)
    platform_qpos = np.array(platform_spec.keys[0].qpos)
else:
    platform_ctrl = np.array([])
    platform_qpos = np.array([])

In [9]:
attachment_frame = scene.worldbody.add_frame()
attachment_frame.attach_body(platform_spec.body("base"), f"{PLATFORM_NAME}_")


In [10]:
# scene.body(f"{ROBOT_NAME}_base").name = f"{ROBOT_NAME}"

In [11]:
print_body_tree(scene.worldbody, scene)
# print_xml(scene.to_xml())

Body Tree for "world"
-Target
-b2_base (Joints: b2_floating_base-FREE)
--b2_FL_hip (Joints: b2_FL_hip_joint-HINGE[Actuator: b2_pos_FL_hip_joint])
---b2_FL_thigh (Joints: b2_FL_thigh_joint-HINGE[Actuator: b2_pos_FL_thigh_joint])
----b2_FL_calf (Joints: b2_FL_calf_joint-HINGE[Actuator: b2_pos_FL_calf_joint])
-----b2_FL_foot
----b2_FL_thigh_protect
--b2_FR_hip (Joints: b2_FR_hip_joint-HINGE[Actuator: b2_pos_FR_hip_joint])
---b2_FR_thigh (Joints: b2_FR_thigh_joint-HINGE[Actuator: b2_pos_FR_thigh_joint])
----b2_FR_calf (Joints: b2_FR_calf_joint-HINGE[Actuator: b2_pos_FR_calf_joint])
-----b2_FR_foot
----b2_FR_thigh_protect
--b2_RL_hip (Joints: b2_RL_hip_joint-HINGE[Actuator: b2_pos_RL_hip_joint])
---b2_RL_thigh (Joints: b2_RL_thigh_joint-HINGE[Actuator: b2_pos_RL_thigh_joint])
----b2_RL_calf (Joints: b2_RL_calf_joint-HINGE[Actuator: b2_pos_RL_calf_joint])
-----b2_RL_foot
----b2_RL_thigh_protect
--b2_RR_hip (Joints: b2_RR_hip_joint-HINGE[Actuator: b2_pos_RR_hip_joint])
---b2_RR_thigh (Joints:

In [12]:
if ROBOT is not None:
    robot_spec_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{ROBOT}.xml")
    robot_spec_spec.copy_during_attach = True
    if len(robot_spec_spec.keys) > 0:
        robot_ctrl = np.array(robot_spec_spec.keys[0].ctrl)
        robot_qpos = np.array(robot_spec_spec.keys[0].qpos)
    else:
        robot_ctrl = np.array([])
        robot_qpos = np.array([])   
        
    attachment_frame = scene.body(f"{PLATFORM_NAME}_base").add_frame(pos=[0, 0, 0.09], axisangle=[1, 0, 0, 0])
    attachment_frame.attach_body(robot_spec_spec.body("base"), f'{ROBOT_NAME}_')
    scene.body(f"{ROBOT_NAME}_base").name = f"{ROBOT_NAME}"
    

In [13]:
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-b2_base (Joints: b2_floating_base-FREE)
--b2_FL_hip (Joints: b2_FL_hip_joint-HINGE[Actuator: b2_pos_FL_hip_joint])
---b2_FL_thigh (Joints: b2_FL_thigh_joint-HINGE[Actuator: b2_pos_FL_thigh_joint])
----b2_FL_calf (Joints: b2_FL_calf_joint-HINGE[Actuator: b2_pos_FL_calf_joint])
-----b2_FL_foot
----b2_FL_thigh_protect
--b2_FR_hip (Joints: b2_FR_hip_joint-HINGE[Actuator: b2_pos_FR_hip_joint])
---b2_FR_thigh (Joints: b2_FR_thigh_joint-HINGE[Actuator: b2_pos_FR_thigh_joint])
----b2_FR_calf (Joints: b2_FR_calf_joint-HINGE[Actuator: b2_pos_FR_calf_joint])
-----b2_FR_foot
----b2_FR_thigh_protect
--b2_RL_hip (Joints: b2_RL_hip_joint-HINGE[Actuator: b2_pos_RL_hip_joint])
---b2_RL_thigh (Joints: b2_RL_thigh_joint-HINGE[Actuator: b2_pos_RL_thigh_joint])
----b2_RL_calf (Joints: b2_RL_calf_joint-HINGE[Actuator: b2_pos_RL_calf_joint])
-----b2_RL_foot
----b2_RL_thigh_protect
--b2_RR_hip (Joints: b2_RR_hip_joint-HINGE[Actuator: b2_pos_RR_hip_joint])
---b2_RR_thigh (Joints:

In [14]:
for s in scene.sites:
    if s.name != "":
        print(s.name)

Target
b2_base_site
z1_x
z1_y
z1_z
z1_flange
z1_TCP


In [15]:
for actuator in scene.actuators:
  print(actuator.name)

b2_pos_FL_hip_joint
b2_pos_FL_thigh_joint
b2_pos_FL_calf_joint
b2_pos_FR_hip_joint
b2_pos_FR_thigh_joint
b2_pos_FR_calf_joint
b2_pos_RL_hip_joint
b2_pos_RL_thigh_joint
b2_pos_RL_calf_joint
b2_pos_RR_hip_joint
b2_pos_RR_thigh_joint
b2_pos_RR_calf_joint
z1_pos_joint1
z1_pos_joint2
z1_pos_joint3
z1_pos_joint4
z1_pos_joint5
z1_pos_joint6
z1_gripper


In [16]:
if CAMERA is not None:
    object_spec = mujoco.MjSpec.from_file(MODEL_PATH + "realsense_d435i.xml")
    object_spec.copy_during_attach = True
    attachment_frame = scene.worldbody.add_frame(pos=[1.0, 0.2, 0.6])
    attachment_frame.attach_body(object_spec.body("cam"), "RS_")

In [17]:
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-b2_base (Joints: b2_floating_base-FREE)
--b2_FL_hip (Joints: b2_FL_hip_joint-HINGE[Actuator: b2_pos_FL_hip_joint])
---b2_FL_thigh (Joints: b2_FL_thigh_joint-HINGE[Actuator: b2_pos_FL_thigh_joint])
----b2_FL_calf (Joints: b2_FL_calf_joint-HINGE[Actuator: b2_pos_FL_calf_joint])
-----b2_FL_foot
----b2_FL_thigh_protect
--b2_FR_hip (Joints: b2_FR_hip_joint-HINGE[Actuator: b2_pos_FR_hip_joint])
---b2_FR_thigh (Joints: b2_FR_thigh_joint-HINGE[Actuator: b2_pos_FR_thigh_joint])
----b2_FR_calf (Joints: b2_FR_calf_joint-HINGE[Actuator: b2_pos_FR_calf_joint])
-----b2_FR_foot
----b2_FR_thigh_protect
--b2_RL_hip (Joints: b2_RL_hip_joint-HINGE[Actuator: b2_pos_RL_hip_joint])
---b2_RL_thigh (Joints: b2_RL_thigh_joint-HINGE[Actuator: b2_pos_RL_thigh_joint])
----b2_RL_calf (Joints: b2_RL_calf_joint-HINGE[Actuator: b2_pos_RL_calf_joint])
-----b2_RL_foot
----b2_RL_thigh_protect
--b2_RR_hip (Joints: b2_RR_hip_joint-HINGE[Actuator: b2_pos_RR_hip_joint])
---b2_RR_thigh (Joints:

In [18]:
# print_xml(scene.to_xml())

Redefine MJCF keys. Delete all keys and define Key 0 as the initial configuration

In [19]:
_tmp = scene.to_xml()
while len(scene.keys)>1:
    scene.delete(scene.keys[-1])
for k in scene.keys:
    k.ctrl = np.concatenate([platform_ctrl, robot_ctrl, np.zeros(len(scene.actuators) - len(platform_ctrl) - len(robot_ctrl))])

Save MJCF scene to XML

In [20]:
if ROBOT is None:
    scene.modelname = f"{PLATFORM_NAME}_scene"
else:
    scene.modelname = f"{PLATFORM_NAME}_{ROBOT_NAME}_scene"
scene.option.timestep = 0.001
with open(MODEL_PATH + f"{PLATFORM}_scene.xml", "w") as f:
    f.write(scene.to_xml())

## Scene and robot definition verification

In [21]:
# from robotblockset.mujoco.robots_pymujoco import mujoco_scene, hc20
# scene = mujoco_scene(MODEL_PATH + f"{ROBOT}_scene.xml", show_camera=None)